# Prepared Remarks vs Q&A Answers — Naive Bayes from Excel

Use the DF1 and DF2 sheets in your earnings-call workbook. No JSON conversion is required.

This notebook implements multinomial Naive Bayes directly with math and Counter; no sklearn or NumPy is required. It groups the train/test split by company (seed 42, approximately 80/20 by company), builds the vocabulary from training data only, and evaluates sentences from held-out companies.

**Important:** PreparedOrQA identifies sections, not speaker roles. The automatic answer selection below is a conservative teaching heuristic, not verified ground truth. Reported scores measure agreement with these provisional labels. Inspect Selection_Review before interpreting accuracy as genuine performance.

For reviewed data, optionally add a ModelLabel column to DF2: `prepared`, `qa`, or `exclude`. A blank cell uses automatic selection. Use `qa` only for management answers, not analyst questions, operator instructions, or closing remarks. Corrections apply to the entire speaker turn. Correct company names in DF1 so aliases do not cross the train/test boundary.

Python random.Random(42) is used for the company split. The split differs from the previous library-based version even with the same seed, so accuracy numbers are not directly comparable. pandas and openpyxl are still used for Excel input/output and tables.


In [ ]:
# Run once if needed: %pip install pandas openpyxl
# Standard-library tools handle paths, text patterns, logarithms, random splitting, and word counts.
from pathlib import Path
import re
import math
import random
from collections import Counter
# pandas handles Excel tables; it does not train the classifier.
import pandas as pd
# Use rich notebook tables when available; otherwise fall back to plain printing.
try:
    from IPython.display import display
except ImportError:
    display = print

# Change this path if your Excel file is elsewhere.
INPUT_FILE = Path(r"C:\Users\walls\Projects\Text Mining\earnings_calls\EarningsCalls.xlsx")
# If the configured file is missing, try the two expected filenames in the current folder.
if not INPUT_FILE.exists():
    INPUT_FILE = next((p for p in [Path("EarningsCalls.xlsx"), Path("EarningsCalls(1).xlsx")]
                       if p.exists()), INPUT_FILE)
if not INPUT_FILE.exists():
    raise FileNotFoundError("Set INPUT_FILE to the location of your earnings-call Excel file.")
# Write results to a separate workbook beside the input. The seed makes the split repeatable.
OUTPUT_FILE = INPUT_FILE.with_name("NaiveBayes_Math_Results.xlsx")
SEED = 42


## 1. Read Excel and connect the two tables


In [ ]:
# DF1 contains call/company information; DF2 contains individual speaker turns.
book = pd.ExcelFile(INPUT_FILE)
calls = pd.read_excel(book, sheet_name="DF1", keep_default_na=False)
turns = pd.read_excel(book, sheet_name="DF2", keep_default_na=False)
# Check required columns before joining the tables.
assert {"EarningsCallID", "CompanyName"}.issubset(calls.columns)
assert {"EarningsCallID", "SpeakerTurnID", "SpeakerName", "PreparedOrQA", "Text"}.issubset(turns.columns)
# IDs must identify unique calls and turns; every turn must belong to a known call.
assert calls["EarningsCallID"].is_unique and turns["SpeakerTurnID"].is_unique
assert turns["EarningsCallID"].isin(calls["EarningsCallID"]).all()

# Rejoin any continuation columns created for Excel's cell-length limit.
text_columns = ["Text"] + sorted(
    [c for c in turns.columns if re.fullmatch(r"Text_\d+", str(c))],
    key=lambda c: int(c.split("_")[-1]))
turns["Text"] = turns[text_columns].fillna("").astype(str).agg("".join, axis=1)
# Add the company name to every turn and preserve chronological turn order.
turns = turns.merge(calls[["EarningsCallID", "CompanyName"]], on="EarningsCallID", validate="many_to_one")
turns = turns.sort_values(["EarningsCallID", "SpeakerTurnID"]).reset_index(drop=True)
# Normalize labels and lookup keys so capitalization and extra spaces do not create false differences.
turns["Section"] = turns["PreparedOrQA"].astype(str).str.strip().str.lower()
assert turns["Section"].isin(["prepared", "qa"]).all(), "Unexpected section label."
turns["SpeakerKey"] = turns["SpeakerName"].astype(str).str.casefold().str.strip()
turns["CompanyKey"] = turns["CompanyName"].astype(str).str.casefold().str.replace(r"\s+", " ", regex=True).str.strip()
assert turns["CompanyKey"].ne("").all(), "Missing company names must be corrected."

# Calls flagged for boundary review will be excluded by the automatic selection rules.
review_calls = set()
if "QA_Review" in book.sheet_names:
    review = pd.read_excel(book, sheet_name="QA_Review")
    review_calls = set(review["EarningsCallID"].dropna())
# Optional human labels override automatic selection later; blank values keep the automatic decision.
if "ModelLabel" not in turns:
    turns["ModelLabel"] = ""
turns["ModelLabel"] = turns["ModelLabel"].fillna("").astype(str).str.strip().str.lower()
assert turns["ModelLabel"].isin(["", "prepared", "qa", "exclude"]).all(), "Invalid ModelLabel."
print("Calls:", len(calls), "Speaker turns:", len(turns))


## 2. Select provisional training examples

A named speaker with at least 80 words in the prepared section is treated as a possible management/IR speaker within that call. Other named speakers are NOT automatically treated as analysts. Their question-like QA turns only provide a provisional question cue. Keep management QA turns only after such a cue; exclude operators, ambiguous speakers, and identifiable closing remarks. Management who speak only in QA may be missed. Review labels can include these cases explicitly.

Calls in QA_Review are excluded automatically unless a turn has a manual ModelLabel. This avoids silently using unresolved boundaries as training truth.


In [ ]:
# Count alphabetic sequences for the speaker-role heuristic; this is not the model tokenizer.
def word_count(text):
    return len(re.findall(r"[A-Za-z]+", text))

# Generic speaker names cannot reliably identify a specific management participant.
def ambiguous_speaker(name):
    return bool(re.search(r"operator|unknown|unidentified|^analyst$|^executive$", name, re.I))

# These patterns detect likely closing remarks and question cues; they are imperfect rules.
closing = re.compile(r"(?:concludes? (?:our|the|this|today.s) (?:call|conference|q&a)|"
                     r"no further questions|back .{0,60}closing remarks|"
                     r"(?:turn|hand).{0,80}(?:closing|final) (?:remarks|comments))", re.I)
question_cue = re.compile(r"\?|\b(?:my question|two questions|first question|could you|can you|would you)\b", re.I)
turns["AutoLabel"] = "exclude"
turns["SelectionReason"] = "Not selected"

# Infer management roles separately within each call, not across the entire dataset.
for call_id, group in turns.groupby("EarningsCallID", sort=False):
    prepared = group[group["Section"].eq("prepared")]
    # Add up each speaker's prepared-section words across all their turns.
    totals = prepared.groupby("SpeakerKey")["Text"].apply(lambda texts: sum(word_count(t) for t in texts))
    managers = {name for name, count in totals.items() if count >= 80 and not ambiguous_speaker(name)}
    # Track whether a plausible external question has opened an answer segment.
    answer_context = False
    # Once a closing cue is detected, stop automatically selecting answers for this call.
    ended = False
    for idx, row in group.iterrows():
        text, name = row["Text"], row["SpeakerKey"]
        label, reason = "exclude", "Speaker role or answer context uncertain"
        if call_id in review_calls:
            reason = "Call requires section-boundary review"
        elif not text.strip():
            reason = "Empty text"
        elif row["Section"] == "prepared":
            if name in managers:
                label, reason = "prepared", "Provisional management prepared turn"
        else:
            # A closing cue near the start of a turn ends answer selection.
            if closing.search(text[:500]):
                ended = True
            if ended:
                reason = "Closing segment"
            # An operator turn resets the answer context.
            elif "operator" in name:
                answer_context = False
                reason = "Operator instruction"
            # External question turns supply context but are not included as training examples.
            elif name not in managers:
                answer_context = not ambiguous_speaker(name) and bool(question_cue.search(text))
                reason = "Possible external question; excluded" if answer_context else reason
            # Consecutive inferred-management turns may be retained as answers in this context.
            elif answer_context:
                label, reason = "qa", "Provisional management answer after a question cue"
        turns.loc[idx, ["AutoLabel", "SelectionReason"]] = [label, reason]

# Apply manual turn-level labels last, including overrides for calls flagged for review.
turns["FinalLabel"] = turns["ModelLabel"].where(turns["ModelLabel"].ne(""), turns["AutoLabel"])
turns["LabelSource"] = turns["ModelLabel"].map(lambda value: "manual" if value else "automatic / provisional")
# Only the two target classes are used for training and evaluation.
selected = turns[turns["FinalLabel"].isin(["prepared", "qa"])].copy()
assert set(selected["FinalLabel"]) == {"prepared", "qa"}, "Both classes are required."
display(pd.crosstab(turns["FinalLabel"], turns["LabelSource"]))


## 3. Split by company, then create sentences

Split companies before fitting the vocabulary. Company aliases must be standardized in DF1. Sentences shorter than four tokens are excluded. Exact normalized duplicates in the test set that also appear in training are removed and counted. This does not remove paraphrases.


In [ ]:
# Shuffle company names using Python's standard library.
companies = sorted(selected["CompanyKey"].unique())
assert len(companies) >= 2, "At least two companies are required."
random.Random(SEED).shuffle(companies)
# Reserve approximately 20% of companies for testing, with at least one company on each side.
test_count = min(len(companies) - 1, max(1, math.ceil(len(companies) * 0.2)))
test_companies = set(companies[:test_count])
selected["Split"] = selected["CompanyKey"].map(lambda company: "test" if company in test_companies else "train")
# Verify that no company or call appears in both sets.
assert not (set(selected.loc[selected.Split.eq("train"), "CompanyKey"]) & set(selected.loc[selected.Split.eq("test"), "CompanyKey"]))
assert not (set(selected.loc[selected.Split.eq("train"), "EarningsCallID"]) & set(selected.loc[selected.Split.eq("test"), "EarningsCallID"]))

# Lowercase English words; retain internal apostrophes. Numbers and punctuation are not features.
def tokenize(text):
    return re.findall(r"[a-z]+(?:'[a-z]+)?", text.lower().replace("’", "'"))

# Protect a few common abbreviations before splitting at punctuation or paragraph breaks.
# This is a simple sentence splitter; unusual abbreviations can still produce imperfect boundaries.
def sentences(text):
    protected = re.sub(r"\b(?:U\.S\.|U\.N\.|Mr\.|Mrs\.|Dr\.)", lambda m: m[0].replace(".", "∯"), text)
    return [s.replace("∯", ".").strip() for s in re.split(r'(?<=[.!?])\s+|\n\s*\n', protected)
            if len(tokenize(s.replace("∯", "."))) >= 4]

# Expand selected turns into sentence rows. Each sentence inherits its turn label and company split.
samples = selected[["EarningsCallID", "SpeakerTurnID", "CompanyKey", "FinalLabel", "Split", "Text"]].copy()
samples["Sentence"] = samples["Text"].map(sentences)
samples = samples.explode("Sentence").dropna(subset=["Sentence"]).drop(columns="Text").reset_index(drop=True)
samples["Normalized"] = samples["Sentence"].map(lambda s: " ".join(tokenize(s)))
# Compare normalized text to remove exact train/test overlap; this does not detect paraphrases.
train_keys = set(samples.loc[samples.Split.eq("train"), "Normalized"])
duplicate_test = samples.Split.eq("test") & samples.Normalized.isin(train_keys)
print("Test sentences removed because they duplicate training:", int(duplicate_test.sum()))
samples = samples.loc[~duplicate_test].copy()
train_data = samples[samples.Split.eq("train")]
test_data = samples[samples.Split.eq("test")]
# Stop rather than report misleading metrics if a split has lost one of the target classes.
for subset in [train_data, test_data]:
    assert set(subset["FinalLabel"]) == {"prepared", "qa"}, "Both classes must appear in each split. Review the dataset."
print("Companies by split:", selected.groupby("Split")["CompanyKey"].nunique().to_dict())
display(pd.crosstab(samples["Split"], samples["FinalLabel"]))


## 4. Calculate Naive Bayes directly with math and Counter

No machine-learning library is used. Count words in training sentences separately for each class. With alpha = 1 (Laplace smoothing):

`P(word | class) = (word_count + alpha) / (total_class_words + alpha * vocabulary_size)`

`score(class) = log(0.5) + sum(word_frequency * log(P(word | class)))`

The class with the larger score wins. Unseen words are ignored. Empty/unknown input and exact numerical ties are marked unclassified, including during evaluation. Probabilities are normalized scores, not calibrated confidence.

Metrics are calculated directly from counts. Unclassified predictions count as errors and reduce recall. Results describe provisional labels unless you have manually reviewed them.


In [ ]:
LABELS = ("prepared", "qa")
# Laplace smoothing adds one to every vocabulary word count in each class.
ALPHA = 1.0

# A. Learn word counts from training sentences only.
def train_naive_bayes(data, alpha=1.0):
    if alpha <= 0:
        raise ValueError("alpha must be positive")
    # Counter stores token frequencies, so repeated words contribute multiple times.
    counts = {label: Counter() for label in LABELS}
    for row in data.itertuples():
        counts[row.FinalLabel].update(tokenize(row.Sentence))
    # Use the union of training words only; test text must not expand this vocabulary.
    vocabulary = set(counts["prepared"]) | set(counts["qa"])
    # The denominator uses the total number of tokens in each class, not distinct words.
    totals = {label: sum(counts[label].values()) for label in LABELS}
    if not vocabulary or any(total == 0 for total in totals.values()):
        raise ValueError("Both classes need training words.")
    return {"counts": counts, "totals": totals, "vocabulary": vocabulary, "alpha": alpha}

# B. Calculate the two class scores for one sentence.
def classify(text, model):
    tokens = tokenize(text)
    # Ignore words absent from training. Count the remaining words in this input.
    known = Counter(word for word in tokens if word in model["vocabulary"])
    # Equal priors give both classes the same starting score, regardless of class frequency.
    scores = {label: math.log(0.5) for label in LABELS}
    contributions = []
    for word, frequency in known.items():
        detail = {"Word": word, "Count": frequency}
        for label in LABELS:
            # Estimate P(word | class), then add its frequency-weighted log probability.
            probability = (model["counts"][label][word] + model["alpha"]) / (
                model["totals"][label] + model["alpha"] * len(model["vocabulary"]))
            detail[label + "_log_contribution"] = frequency * math.log(probability)
            scores[label] += detail[label + "_log_contribution"]
        contributions.append(detail)
    # Subtract the maximum score before exponentiating to avoid underflow.
    maximum = max(scores.values())
    weights = {label: math.exp(scores[label] - maximum) for label in LABELS}
    # Normalize the two positive weights to sum to one; these are not calibrated confidence values.
    probabilities = {label: weights[label] / sum(weights.values()) for label in LABELS}
    # Treat score differences within 1e-12 as a numerical tie.
    tie = math.isclose(scores["prepared"], scores["qa"], rel_tol=0, abs_tol=1e-12)
    prediction = "unclassified" if not known or tie else max(scores, key=scores.get)
    return {"label": prediction, "scores": scores, "probabilities": probabilities,
            "known_words": sum(known.values()),
            "unknown_words": [word for word in tokens if word not in model["vocabulary"]],
            "contributions": contributions}

model = train_naive_bayes(train_data, ALPHA)
print("Training word counts:", model["totals"])
print("Vocabulary size:", len(model["vocabulary"]))

# C. Predict held-out sentences and calculate evaluation metrics.
prediction = [classify(text, model)["label"] for text in test_data["Sentence"]]
# Matrix rows are reference labels; columns are predictions, including unclassified cases.
confusion = pd.DataFrame(0, index=LABELS, columns=[*LABELS, "unclassified"])
for actual, predicted in zip(test_data["FinalLabel"], prediction):
    confusion.loc[actual, predicted] += 1

rows = []
for label in LABELS:
    # Precision: correct predictions / predictions of this class.
    # Recall: correct predictions / actual examples of this class (including unclassified errors).
    true_positive = int(confusion.loc[label, label])
    actual_count = int(confusion.loc[label].sum())
    predicted_count = int(confusion[label].sum())
    precision = true_positive / predicted_count if predicted_count else 0.0
    recall = true_positive / actual_count if actual_count else 0.0
    # F1 is the harmonic mean of precision and recall; zero denominators return zero.
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    rows.append({"Metric": label, "precision": precision, "recall": recall,
                 "f1-score": f1, "support": actual_count})
report = pd.DataFrame(rows).set_index("Metric")
# Accuracy weights sentences equally; balanced accuracy gives each class equal weight.
accuracy = sum(int(confusion.loc[label, label]) for label in LABELS) / len(test_data)
balanced_accuracy = sum(report["recall"]) / len(LABELS)
# Baseline: always predict the most common training label, then score that rule on the test set.
training_majority = train_data["FinalLabel"].value_counts().idxmax()
baseline = float(test_data["FinalLabel"].eq(training_majority).mean())
print("Accuracy:", round(accuracy, 4))
print("Balanced accuracy:", round(balanced_accuracy, 4))
print("Training-majority baseline:", round(baseline, 4))
print("Unclassified test sentences:", int(confusion["unclassified"].sum()))
print("Confusion matrix: actual rows, predicted columns")
display(confusion)
display(report)
# Keep identifiers and sentence text so errors can be traced back to their source turns.
test_results = test_data.drop(columns="Normalized").copy()
test_results["Predicted"] = prediction
test_results["Correct"] = test_results["FinalLabel"].eq(prediction)
display(test_results.loc[~test_results.Correct].head(10))

# D. Verify that smoothed word probabilities sum to one.
for label in LABELS:
    probability_sum = sum((model["counts"][label][word] + ALPHA) /
                          (model["totals"][label] + ALPHA * len(model["vocabulary"]))
                          for word in model["vocabulary"])
    assert math.isclose(probability_sum, 1.0, abs_tol=1e-10)
assert classify("", model)["label"] == "unclassified"
print("Probability checks passed.")


## 5. Classify a new sentence and inspect word scores

Edit new_sentence and rerun this cell. Probabilities are model outputs, not calibrated confidence. Empty or entirely unseen input is left unclassified.


In [ ]:
# Change this sentence to inspect a new prediction after running the earlier cells.
new_sentence = "Thanks for the question. Let me explain why revenue increased."
result = classify(new_sentence, model)
print("Prediction:", result["label"])
print("Log scores:", result["scores"])
print("Normalized model probabilities:", result["probabilities"])
print("Recognized words:", result["known_words"])
print("Ignored words:", result["unknown_words"])
# Each row shows one known word, its frequency, and its contribution to each class log score.
display(pd.DataFrame(result["contributions"]))


## 6. Save results for review

This creates a separate NaiveBayes_Math_Results.xlsx; it never overwrites the input file. Selection_Review contains every original turn, automatic selection, final label, reason, and split. To correct labels, add ModelLabel to the original DF2 sheet, match rows by SpeakerTurnID, enter prepared / qa / exclude, and rerun the notebook. Review the full turn, including continuation columns. The results workbook itself is not an input workbook.


In [ ]:
from openpyxl.styles import Alignment
from openpyxl.utils import get_column_letter

# Preserve long strings by continuing them in additional columns.
def excel_safe(data):
    # Work on a copy so Excel formatting does not change the analysis tables.
    data = data.copy()
    for column in list(data.columns):
        if data[column].dtype == object:
            lengths = data[column].map(lambda v: len(v) if isinstance(v, str) else 0)
            # Use chunks of 30,000 characters to stay below Excel's per-cell text limit.
            if len(lengths) and lengths.max() > 30000:
                text = data[column].fillna("").astype(str)
                for start in range(0, int(lengths.max()), 30000):
                    name = column if start == 0 else f"{column}_{start // 30000 + 1}"
                    data[name] = text.str.slice(start, start + 30000)
    return data

# Retain every original turn for review. Excluded turns have no train/test split.
selection_review = turns[["EarningsCallID", "SpeakerTurnID", "CompanyName", "SpeakerName", "Section", "Text", "AutoLabel", "ModelLabel", "FinalLabel", "SelectionReason", "LabelSource"]].merge(
    selected[["SpeakerTurnID", "Split"]], on="SpeakerTurnID", how="left", validate="one_to_one")
# Include interpretation and correction instructions in the exported workbook.
notes = pd.DataFrame({"Note": [
    "Automatic labels are provisional. Inspect Selection_Review before relying on evaluation scores.",
    "Company split: seed 42; approximately 80/20 by company, not by sentence.",
    "Classes: management prepared remarks vs management answers. Automatic selection may omit or misidentify speakers.",
    "Long text continues in numbered columns. Rejoin without separators.",
    "Correct ModelLabel in the original input DF2 by SpeakerTurnID; the results file is not an input workbook.",
]})
# Save metrics, the confusion matrix, selection decisions, and sentence predictions as separate sheets.
# Re-running this cell replaces the results workbook at OUTPUT_FILE.
with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
    for name, frame in {"Notes": notes, "Metrics": report.reset_index(names="Metric"),
                        "Overall": pd.DataFrame({"Metric": ["Accuracy", "Balanced accuracy", "Majority baseline"], "Value": [accuracy, balanced_accuracy, baseline]}),
                        "Confusion": confusion.reset_index(names="Actual"),
                        "Selection_Review": selection_review, "Test_Predictions": test_results}.items():
        excel_safe(frame).to_excel(writer, sheet_name=name, index=False)
        ws = writer.sheets[name]
        # Keep headers visible, enable filtering, and format text for easier reading.
        ws.freeze_panes = "A2"
        ws.auto_filter.ref = ws.dimensions
        for i in range(1, ws.max_column + 1):
            ws.column_dimensions[get_column_letter(i)].width = 30
        for row in ws.iter_rows(min_row=2):
            ws.row_dimensions[row[0].row].height = 45
            for cell in row:
                cell.alignment = Alignment(wrap_text=True, vertical="top")
                # Store transcript strings as text so leading equals signs are not treated as formulas.
                if isinstance(cell.value, str):
                    cell.data_type = "s"
print("Saved:", OUTPUT_FILE)
